# 🧪 Eksperimen USD/IDR Forecasting — Full Pipeline
## Rupiah Resilience

Notebook ini menjalankan seluruh pipeline eksperimen:
1. **Hyperparameter Optimization** (Optuna)
2. **Training & Benchmarking** (3 skenario × semua model)
3. **Evaluasi & Metrik**
4. **Future Forecasting** (prediksi Juni 2023 - Mei 2026)

**Cara pakai:** Run All dari Cell 1 ke bawah.

## Cell 1 — Setup & Clone Repo

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Clone repo dari GitHub
# ══════════════════════════════════════════════════════════════
import os

REPO_URL = "https://github.com/ruwwww/fp-pap.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"Cloned: {REPO_NAME}")
else:
    print(f"Repo sudah ada: {REPO_NAME}")

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")
!ls

## Cell 2 — Install Dependencies

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Install semua dependencies
# ══════════════════════════════════════════════════════════════
!pip install -q -r requirements.txt
!pip install -q optuna lightgbm

import sys
print(f"Python: {sys.version}")

# Verifikasi
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Cell 3 — Tahap 1a: Hyperparameter Search — ML Models

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: Optuna Search — ML Models (50 trials, 4 jobs)
# Model: RandomForest, XGBoost, LightGBM, GradientBoosting, Ridge
# Estimasi: ~5-10 menit
# ══════════════════════════════════════════════════════════════
!python scripts/search.py --category ML --trials 50 --jobs 4

## Cell 4 — Tahap 1b: Hyperparameter Search — DL Models

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Optuna Search — DL Models (20 trials)
# Model: LSTM, GRU, CNN-LSTM
# Estimasi: ~10-20 menit (GPU)
# ══════════════════════════════════════════════════════════════
!python scripts/search.py --category DL --trials 20

## Cell 5 — Tahap 2: Training & Benchmarking (3 Skenario)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: Training semua model × 3 skenario
# Skenario: 80/20, 70/30, 60/40
# Estimasi: ~10-15 menit
# ══════════════════════════════════════════════════════════════
!python scripts/train.py --data data/raw/data_train.csv --use-best-params --parallel --workers 4

## Cell 6 — Tahap 3: Evaluasi & Generate Laporan

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: Generate tabel rekapitulasi, grafik, dan laporan
# ══════════════════════════════════════════════════════════════
!python scripts/evaluate.py --report-dir report/

## Cell 7 — Tahap 4: Future Forecasting (Prediksi Masa Depan)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7: Forecast USD/IDR Juni 2023 — Mei 2026
# Output: data/submissions/forecast.csv
# ══════════════════════════════════════════════════════════════
!python scripts/forecast.py --model ensemble --use-best-params

## Cell 8 — Lihat Hasil & Tabel Perbandingan

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: Tampilkan tabel perbandingan hasil
# ══════════════════════════════════════════════════════════════
import pandas as pd
from IPython.display import display, Markdown

results_path = "results/metrics/results_summary.csv"
try:
    df = pd.read_csv(results_path)
    print("=" * 70)
    print("REKAPITULASI HASIL")
    print("=" * 70)
    display(df.sort_values(["Scenario", "RMSE"]).reset_index(drop=True))

    print("\n" + "=" * 70)
    print("BEST MODEL PER SKENARIO")
    print("=" * 70)
    best = df.loc[df.groupby("Scenario")["RMSE"].idxmin()]
    display(best[["Scenario", "Model", "RMSE", "MAPE (%)", "MAE", "R\u00b2"]])
except FileNotFoundError:
    print("results_summary.csv belum ada. Jalankan Cell 5 & 6 terlebih dahulu.")

## Cell 9 — Lihat Plot Hasil

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: Tampilkan plot hasil
# ══════════════════════════════════════════════════════════════
import os
from IPython.display import display, Image
import glob

plots_dir = "results/plots"
if os.path.exists(plots_dir):
    plot_files = sorted(glob.glob(f"{plots_dir}/*.png"))
    print(f"Found {len(plot_files)} plots:\n")
    for pf in plot_files:
        name = os.path.basename(pf)
        print(f"--- {name} ---")
        display(Image(filename=pf, width=900))
        print()
else:
    print("plots/ belum ada. Jalankan pipeline terlebih dahulu.")

## Cell 10 — Download Artifacts

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10: Download semua artifacts sebagai ZIP
# ══════════════════════════════════════════════════════════════
import shutil
from google.colab import files

# Zip results
shutil.make_archive("experiment_artifacts", "zip", ".", "results")
print("Zipped: experiment_artifacts.zip")

# Zip runs
if os.path.exists("runs"):
    shutil.make_archive("runs_artifacts", "zip", ".", "runs")
    print("Zipped: runs_artifacts.zip")

# Download
files.download("experiment_artifacts.zip")
if os.path.exists("runs_artifacts.zip"):
    files.download("runs_artifacts.zip")
if os.path.exists("data/submissions/forecast.csv"):
    files.download("data/submissions/forecast.csv")

print("\nDone! Semua artifacts sudah ter-download.")